# Figure 1 — FINAL reproducible Colab

Ejecuta **Runtime → Run all**. Si falta el ZIP de Phase IV o alguna de las tres imágenes, Colab pedirá subirla. Acepta `tribal-woman` o `tribal-women`. Exporta PNG 600 dpi, PDF y SVG.


In [ ]:
from pathlib import Path
import zipfile, shutil
import numpy as np, pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib import gridspec
from scipy.ndimage import gaussian_filter
try:
    from google.colab import files
    IN_COLAB=True
except Exception:
    IN_COLAB=False

BASE=Path("/content") if IN_COLAB else Path("/mnt/data")
ZIP=BASE/"painting_geometry_phase4_artbench_pilot.zip"
IMGDIR=BASE/"figure1_images"; IMGDIR.mkdir(exist_ok=True)
ART=BASE/"artbench_data"
OUT=BASE/"figure1_final_output"; OUT.mkdir(exist_ok=True)
DESC="geom__curv__kappa_ref_s2p0_grad_weighted_abs"
SIGMAS=[1,2,4,8]
PANEL_B_SIGMA=2
Q=[.15,.30,.50,.70,.85]
DPI=600

SEL=[
 dict(role="Low geometry",style="post_impressionism",artist="anita-malfatti",
      label="Anita Malfatti",title="Fernanda de Castro",year="1922",
      csv=["anita-malfatti_fernanda-de-castro-1922.jpg"],
      img=["anita-malfatti_fernanda-de-castro-1922.jpg"],expected=.252172),
 dict(role="Intermediate geometry",style="post_impressionism",artist="amrita-sher-gil",
      label="Amrita Sher-Gil",title="Tribal Women",year="1938",
      csv=["amrita-sher-gil_tribal-women-1938.jpg","amrita-sher-gil_tribal-woman-1938.jpg"],
      img=["amrita-sher-gil_tribal-women-1938.jpg","amrita-sher-gil_tribal-woman-1938.jpg"],expected=.324556),
 dict(role="High geometry",style="post_impressionism",artist="abraham-manievich",
      label="Abraham Manievich",title="The Yellow House",year="",
      csv=["abraham-manievich_the-yellow-house.jpg"],
      img=["abraham-manievich_the-yellow-house.jpg"],expected=.403162)
]

def resize(im,L=512):
    w,h=im.size; s=L/max(w,h)
    return im.resize((round(w*s),round(h*s)),Image.Resampling.LANCZOS)

def lum(rgb):
    a=np.asarray(rgb,float); y=.299*a[...,0]+.587*a[...,1]+.114*a[...,2]
    y-=y.min(); d=y.max()-y.min()
    return y/d if d>0 else np.zeros_like(y)

def smooth(I,sref,L,ref=512):
    return gaussian_filter(I,sigma=sref*L/ref,mode="reflect",truncate=3)

def curvature(I,sref,L,ref=512,eps=1e-12,q=.20):
    s=sref*L/ref; kw=dict(sigma=s,mode="reflect",truncate=3)
    Ix=gaussian_filter(I,order=(0,1),**kw); Iy=gaussian_filter(I,order=(1,0),**kw)
    Ixx=gaussian_filter(I,order=(0,2),**kw); Iyy=gaussian_filter(I,order=(2,0),**kw)
    Ixy=gaussian_filter(I,order=(1,1),**kw)
    g2=Ix*Ix+Iy*Iy; g=np.sqrt(g2)
    k=(Ixx*Iy*Iy-2*Ix*Iy*Ixy+Iyy*Ix*Ix)/np.power(g2+eps*eps,1.5)
    ok=np.isfinite(k)&np.isfinite(g); pos=g[ok&(g>0)]
    th=np.quantile(pos,q) if pos.size else 0
    return s*k, ok&(g>=th)

def locate(names):
    for n in names:
        for root in (IMGDIR,BASE,ART):
            if root.exists():
                p=root/n
                if p.exists(): return p
                h=list(root.rglob(n))
                if h:return h[0]
    target={Path(n).stem.lower().replace("tribal-women","tribal-woman") for n in names}
    for root in (IMGDIR,BASE,ART):
        if root.exists():
            for p in root.rglob("*"):
                if p.is_file() and p.suffix.lower() in {".jpg",".jpeg",".png",".webp"}:
                    if p.stem.lower().replace("tribal-women","tribal-woman") in target:return p
    return None

def ensure():
    miss=[]
    if not ZIP.exists(): miss.append("painting_geometry_phase4_artbench_pilot.zip")
    for s in SEL:
        if locate(s["img"]) is None: miss.append(" OR ".join(s["img"]))
    if miss and IN_COLAB:
        print("Faltan:\n- "+"\n- ".join(miss)+"\n\nSelecciona los archivos:")
        up=files.upload()
        for n,b in up.items():
            p=BASE/n; p.write_bytes(b)
            if Path(n).suffix.lower() in {".jpg",".jpeg",".png",".webp"}:
                shutil.copy2(p,IMGDIR/n)
    if not ZIP.exists(): raise FileNotFoundError(f"Missing {ZIP}")

def clean(ax):
    ax.set_xticks([]);ax.set_yticks([])
    for sp in ax.spines.values():sp.set_visible(False)

def letter(ax,s):
    ax.text(-.055,1.035,s,transform=ax.transAxes,fontsize=20,fontweight="bold",ha="left",va="bottom")


In [ ]:
ensure()

with zipfile.ZipFile(ZIP) as z:
    name=[n for n in z.namelist() if n.endswith("artbench_pilot_features.csv")][0]
    with z.open(name) as f: phase4_features=pd.read_csv(f)

rows=[]
for s in SEL:
    h=phase4_features[(phase4_features["style"].astype(str)==s["style"])&
                      (phase4_features["artist"].astype(str)==s["artist"])]
    e=h[h["filename"].astype(str).isin(s["csv"])]
    if len(e)==1:h=e
    elif len(e)==0 and s["artist"]=="amrita-sher-gil":
        f=h[h["filename"].astype(str).str.contains(r"tribal-wom(?:an|en)-1938",case=False,regex=True,na=False)]
        if len(f)==1:h=f
    if len(h)!=1: raise RuntimeError(f"No unique Phase-IV row for {s['label']} / {s['title']}; found {len(h)}")
    r=h.iloc[0].to_dict();r.update(label=s["label"],title=s["title"],role=s["role"],year=s["year"])
    rows.append(r)
selected_df=pd.DataFrame(rows)
display(selected_df[["role","label","title","filename",DESC]])

records=[]
for s in SEL:
    p=locate(s["img"])
    if p is None: raise FileNotFoundError(f"Missing image for {s['title']}: {s['img']}")
    im=resize(Image.open(p).convert("RGB"),512); rgb=np.asarray(im); L=lum(rgb)
    r=selected_df[selected_df["label"]==s["label"]].iloc[0]
    records.append({**s,"path":str(p),"rgb":rgb,"L":L,"value":float(r[DESC])})

mid=[r for r in records if r["role"].startswith("Intermediate")][0]
K={};M={}
for s in SIGMAS: K[s],M[s]=curvature(mid["L"],s,max(mid["L"].shape))
clip=float(np.percentile(np.concatenate([np.abs(K[s][M[s]]) for s in SIGMAS]),99))
print("✓ Inputs, Phase IV, images and curvature ready. clip =",clip)


In [ ]:
fig=plt.figure(figsize=(14.4,10.2))
gs=gridspec.GridSpec(3,12,figure=fig,height_ratios=[1.08,1,1.12],hspace=.34,wspace=.18)
fig.suptitle("Figure 1. From painting to multiscale luminance geometry",x=.03,y=.988,ha="left",fontsize=19,fontweight="bold")
fig.text(.03,.951,"Within one style category, paintings occupy markedly different positions in the level-set geometry measured at an intermediate spatial scale.",fontsize=10.8,color="dimgray")

for j,r in enumerate(records):
    ax=fig.add_subplot(gs[0,4*j:4*(j+1)])
    if j==0:letter(ax,"a")
    ax.imshow(r["rgb"]);clean(ax)
    yr=f" ({r['year']})" if r["year"] else " (n.d.)"
    ax.set_title(f"{r['role']}\n{r['label']}, {r['title']}{yr}\n"+rf"$G_{{\sigma=2}}={r['value']:.3f}$",fontsize=9.8,pad=6)

for j,r in enumerate(records):
    ax=fig.add_subplot(gs[1,4*j:4*(j+1)])
    if j==0:letter(ax,"b")
    S=smooth(r["L"],PANEL_B_SIGMA,max(r["L"].shape))
    D=S-S.min();D=D/(D.max()-D.min()) if D.max()>D.min() else D
    levels=np.unique(np.quantile(S,Q))
    ax.imshow(D,cmap="gray",vmin=0,vmax=1,alpha=.96)
    ax.contour(S,levels=levels,colors="white",linewidths=1.05,alpha=.96)
    clean(ax);ax.set_title("Iso-luminance contours — "+r["role"].replace(" geometry","").lower(),fontsize=9.8,pad=4)

sub=gridspec.GridSpecFromSubplotSpec(1,4,subplot_spec=gs[2,:],wspace=.06)
axs=[];ims=[]
for j,s in enumerate(SIGMAS):
    ax=fig.add_subplot(sub[0,j])
    if j==0:letter(ax,"c")
    ax.imshow(np.clip(.86*mid["L"]+.14,0,1),cmap="gray",vmin=0,vmax=1)
    im=ax.imshow(np.ma.masked_where(~M[s],K[s]),cmap="coolwarm",vmin=-clip,vmax=clip,alpha=.78)
    clean(ax);ax.set_title(rf"$\sigma_{{ref}}={s}$",fontsize=10.5,pad=5)
    axs.append(ax);ims.append(im)

left,right=axs[0].get_position().x0,axs[-1].get_position().x1
bottom=min(a.get_position().y0 for a in axs)-.045
cax=fig.add_axes([left+.08,bottom,(right-left)-.16,.016])
cb=plt.colorbar(ims[0],cax=cax,orientation="horizontal")
cb.set_ticks([-clip,0,clip]);cb.set_ticklabels([f"{-clip:.2f}","0",f"{clip:.2f}"])
cb.set_label(r"scale-normalised curvature $\tilde{\kappa}_{\sigma}$",fontsize=9,labelpad=4)
cb.ax.tick_params(labelsize=8.5,length=0);cb.outline.set_linewidth(.6)
fig.subplots_adjust(left=.045,right=.985,top=.90,bottom=.10)

stem=OUT/"Figure1_multiscale_luminance_geometry_FINAL"
fig.savefig(stem.with_suffix(".png"),dpi=DPI,bbox_inches="tight",facecolor="white")
fig.savefig(stem.with_suffix(".pdf"),bbox_inches="tight",facecolor="white")
fig.savefig(stem.with_suffix(".svg"),bbox_inches="tight",facecolor="white")
plt.show()
print("Saved:",stem.with_suffix(".png"),stem.with_suffix(".pdf"),stem.with_suffix(".svg"),sep="\n")
